In [6]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv("spam.csv", encoding="latin-1")
df = df.drop(["Unnamed: 2", "Unnamed: 3", "Unnamed: 4"], axis=1)

In [7]:

df = df.rename({"v1":"is_spam", "v2":"msg"}, axis=1)

In [8]:
def is_spam(s):
    if "ham" in s.lower():
        return 0
    return 1
df["is_spam"] = df["is_spam"].apply(is_spam)

In [14]:
import nltk
from nltk.corpus import stopwords
stop_words = set(stopwords.words("english"))
from nltk.stem import WordNetLemmatizer
lemmitizer = WordNetLemmatizer()

In [17]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))
corpus = []

for i in range(len(df)):
    review = re.sub("[^a-zA-Z]", " ", df["msg"][i])
    review = review.lower().split()
    words = [
        lemmatizer.lemmatize(word)
        for word in review
        if word not in stop_words
    ]
    corpus.append(" ".join(words))

In [19]:
from sklearn.model_selection import train_test_split
df["msg_corpus"] = corpus
X_trainn, X_testt, y_train, y_test = train_test_split(
    df["msg_corpus"],
    df["is_spam"],
    test_size=0.2,
    random_state=42
)

In [20]:
print(df.isnull().sum())

is_spam       0
msg           0
msg_corpus    0
dtype: int64


In [21]:
## TFIDF(better than bagging)
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=1000,
    ngram_range=(1, 3)
)

X_train = tfidf.fit_transform(X_trainn)
X_test = tfidf.transform(X_testt)

In [22]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=32,
    callbacks=[early_stop]
)

Epoch 1/20
112/112 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8692 - loss: 0.3253 - val_accuracy: 0.8711 - val_loss: 0.1865
Epoch 2/20
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9641 - loss: 0.1070 - val_accuracy: 0.9787 - val_loss: 0.0990
Epoch 3/20
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9885 - loss: 0.0385 - val_accuracy: 0.9809 - val_loss: 0.0941
Epoch 4/20
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9930 - loss: 0.0244 - val_accuracy: 0.9798 - val_loss: 0.0996
Epoch 5/20
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9958 - loss: 0.0145 - val_accuracy: 0.9832 - val_loss: 0.1069
Epoch 6/20
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9975 - loss: 0.0115 - val_accuracy: 0.9843 - val_loss: 0.1080
Epoch 7/20
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9980 - loss: 0.0084 - val_accuracy: 0.9832 - val_loss: 0.1134
Epoch 8/20
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9983 - loss: 0.0082 - val_accuracy: 0.

In [23]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy:.4f}")

35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9794 - loss: 0.0808
Test Accuracy: 0.9794


In [24]:
from sklearn.metrics import classification_report, confusion_matrix

y_prob = model.predict(X_test)
y_pred = (y_prob > 0.5).astype(int)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
[[947  10]
 [ 13 144]]
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       957
           1       0.94      0.92      0.93       157

    accuracy                           0.98      1114
   macro avg       0.96      0.95      0.96      1114
weighted avg       0.98      0.98      0.98      1114



In [25]:
df

,is_spam,msg,msg_corpus
0,0,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available bugis n great ...
1,0,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entry wkly comp win fa cup final tkts st ...
3,0,U dun say so early hor... U c already then say...,u dun say early hor u c already say
4,0,"Nah I don't think he goes to usf, he lives aro...",nah think go usf life around though
...,...,...,...
5565,1,This is the 2nd time we have tried 2 contact u...,nd time tried contact u u pound prize claim ea...
5566,0,Will Ì_ b going to esplanade fr home?,b going esplanade fr home
5567,0,"Pity, * was in mood for that. So...any other s...",pity mood suggestion
5568,0,The guy did some bitching but I acted like i'd...,guy bitching acted like interested buying some...


In [ ]:
import re
import numpy as np

while True:
    text = input("\nEnter SMS (or type 'exit'): ")

    if text.lower() == "exit":
        break

    # Same preprocessing as training
    review = re.sub("[^a-zA-Z]", " ", text)
    review = review.lower().split()

    words = [
        lemmatizer.lemmatize(word)
        for word in review
        if word not in stop_words
    ]

    processed = " ".join(words)

    # TF-IDF transform
    X = tfidf.transform([processed])

    # Predict
    prob = model.predict(X, verbose=0)[0][0]

    print(f"Spam Probability: {prob:.4f}")

    if prob >= 0.5:
        print("🚨 Prediction: SPAM")
    else:
        print("✅ Prediction: HAM")


Enter SMS (or type 'exit'):  Hey, where are you? I'm waiting outside.


Spam Probability: 0.0004
✅ Prediction: HAM



Enter SMS (or type 'exit'):  Can you call me when you get home?


Spam Probability: 0.0013
✅ Prediction: HAM



Enter SMS (or type 'exit'):  I'll be 10 minutes late for class.


...

Spam Probability: 0.9855
🚨 Prediction: SPAM



Enter SMS (or type 'exit'):  FREE entry into our weekly lottery. Reply WIN to claim now.


Spam Probability: 0.9888
🚨 Prediction: SPAM



Enter SMS (or type 'exit'):  You have been selected for a FREE iPhone 16. Visit our website immediately.


Spam Probability: 0.8136
🚨 Prediction: SPAM
